Lab 9 NB with Gorge Rodriguez, and Maxwell Colburn, Logan Cheng

kaggle.com/datasets/fedesoriano/stroke-prediction-dataset

In [1]:
# Import required libraries
import pandas as pd                                      
from sklearn.naive_bayes import CategoricalNB            
from sklearn.preprocessing import LabelEncoder           
from sklearn.model_selection import train_test_split     
from sklearn.metrics import (classification_report,      
                             confusion_matrix,           
                             accuracy_score)             

In [2]:
# Load the dataset
df = pd.read_csv("healthcare-dataset-stroke-data.csv")

# Drop the 'id' column
df = df.drop(columns=["id"])

# Drop rows where 'bmi' is missing
df = df.dropna(subset=["bmi"])

# Remove the 'Other' gender category
df = df[df["gender"] != "Other"]

print(f"Total patients loaded : {len(df)}")
print(f"Stroke cases (Yes=1)  : {sum(df['stroke'] == 1)}")
print(f"No stroke cases (No=0): {sum(df['stroke'] == 0)}")
print()

df.head()

Total patients loaded : 4908
Stroke cases (Yes=1)  : 209
No stroke cases (No=0): 4699



,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
2,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1
5,Male,81.0,0,0,Yes,Private,Urban,186.21,29.0,formerly smoked,1


In [3]:
# Bin numeric columns into categories
# Age -> Young (0-40), Middle-aged (41-60), Senior (61+)
df["age_group"] = pd.cut(
    df["age"],
    bins=[0, 40, 60, 120],
    labels=["Young", "Middle-aged", "Senior"]
)

# Average Glucose Level -> Low (<70), Normal (70-140), High (>140)
df["glucose_level"] = pd.cut(
    df["avg_glucose_level"],
    bins=[0, 70, 140, 300],
    labels=["Low", "Normal", "High"]
)

# BMI -> Underweight (<18.5), Normal (18.5-25), Overweight (25-30), Obese (30+)
df["bmi_category"] = pd.cut(
    df["bmi"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"]
)

# Convert hypertension and heart_disease from 0/1 to Yes/No
df["hypertension"]  = df["hypertension"].map({1: "Yes", 0: "No"})
df["heart_disease"] = df["heart_disease"].map({1: "Yes", 0: "No"})

print("Sample of binned columns:")
df[["age", "age_group", "avg_glucose_level", "glucose_level", "bmi", "bmi_category"]].head(8)

Sample of binned columns:


,age,age_group,avg_glucose_level,glucose_level,bmi,bmi_category
0,67.0,Senior,228.69,High,36.6,Obese
2,80.0,Senior,105.92,Normal,32.5,Obese
3,49.0,Middle-aged,171.23,High,34.4,Obese
4,79.0,Senior,174.12,High,24.0,Normal
5,81.0,Senior,186.21,High,29.0,Overweight
6,74.0,Senior,70.09,Normal,27.4,Overweight
7,69.0,Senior,94.39,Normal,22.8,Normal
9,78.0,Senior,58.57,Low,24.2,Normal


In [4]:
# Encode categories as numbers
# These are the columns we'll use to make predictions
feature_columns = [
    "gender",
    "hypertension",
    "heart_disease",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status",
    "age_group",
    "glucose_level",
    "bmi_category"
]

# Keep only the feature columns and the label
df_model = df[feature_columns + ["stroke"]].copy()

# Drop any remaining rows with missing values (from the binning)
df_model = df_model.dropna()

# Create a dictionary to store one encoder per column
encoders = {}

for col in feature_columns:
    enc = LabelEncoder()
    df_model[col + "_num"] = enc.fit_transform(df_model[col].astype(str))
    encoders[col] = enc
    print(f"  {col}: {dict(zip(enc.classes_, enc.transform(enc.classes_)))}")

  gender: {'Female': np.int64(0), 'Male': np.int64(1)}
  hypertension: {'No': np.int64(0), 'Yes': np.int64(1)}
  heart_disease: {'No': np.int64(0), 'Yes': np.int64(1)}
  ever_married: {'No': np.int64(0), 'Yes': np.int64(1)}
  work_type: {'Govt_job': np.int64(0), 'Never_worked': np.int64(1), 'Private': np.int64(2), 'Self-employed': np.int64(3), 'children': np.int64(4)}
  Residence_type: {'Rural': np.int64(0), 'Urban': np.int64(1)}
  smoking_status: {'Unknown': np.int64(0), 'formerly smoked': np.int64(1), 'never smoked': np.int64(2), 'smokes': np.int64(3)}
  age_group: {'Middle-aged': np.int64(0), 'Senior': np.int64(1), 'Young': np.int64(2)}
  glucose_level: {'High': np.int64(0), 'Low': np.int64(1), 'Normal': np.int64(2)}
  bmi_category: {'Normal': np.int64(0), 'Obese': np.int64(1), 'Overweight': np.int64(2), 'Underweight': np.int64(3)}


In [ ]:
# Split into training and test
# Build the numeric feature matrix
num_cols = [col + "_num" for col in feature_columns]
X = df_model[num_cols]
y = df_model["stroke"]

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

Training samples : 3926
Testing samples  : 982


In [6]:
# Train the CategoricalNB model
model = CategoricalNB()       
model.fit(X_train, y_train)



CategoricalNB()

In [7]:
# Evaluate model performance
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print()

print("Confusion Matrix:")
print("(Rows = Actual, Columns = Predicted)")
cm = confusion_matrix(y_test, y_pred)
print(f"                Predicted No  Predicted Yes")
print(f"  Actual No  :      {cm[0][0]}           {cm[0][1]}")
print(f"  Actual Yes :      {cm[1][0]}           {cm[1][1]}")
print()

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["No Stroke", "Stroke"]))

Accuracy: 91.55%

Confusion Matrix:
(Rows = Actual, Columns = Predicted)
                Predicted No  Predicted Yes
  Actual No  :      893           36
  Actual Yes :      47           6

Classification Report:
              precision    recall  f1-score   support

   No Stroke       0.95      0.96      0.96       929
      Stroke       0.14      0.11      0.13        53

    accuracy                           0.92       982
   macro avg       0.55      0.54      0.54       982
weighted avg       0.91      0.92      0.91       982



In [ ]:
# Predicit for new patients
new_patients = [
    {
        "gender": "Male", "hypertension": "Yes", "heart_disease": "Yes",
        "ever_married": "Yes", "work_type": "Private", "Residence_type": "Urban",
        "smoking_status": "smokes", "age_group": "Senior",
        "glucose_level": "High", "bmi_category": "Obese"
    },
    {
        "gender": "Female", "hypertension": "No", "heart_disease": "No",
        "ever_married": "No", "work_type": "Private", "Residence_type": "Rural",
        "smoking_status": "never smoked", "age_group": "Young",
        "glucose_level": "Normal", "bmi_category": "Normal"
    },
    {
        "gender": "Male", "hypertension": "No", "heart_disease": "No",
        "ever_married": "Yes", "work_type": "Self-employed", "Residence_type": "Urban",
        "smoking_status": "formerly smoked", "age_group": "Middle-aged",
        "glucose_level": "Normal", "bmi_category": "Overweight"
    },
]

for i, patient in enumerate(new_patients, 1):
    # Encode each feature
    encoded = []
    for col in feature_columns:
        num = encoders[col].transform([patient[col]])[0]
        encoded.append(num)

    # Wrap in a DataFrame with column names
    single_input = pd.DataFrame([encoded], columns=num_cols)

    prediction  = model.predict(single_input)[0]
    probability = model.predict_proba(single_input)[0]
    result = "YES - Stroke Risk" if prediction == 1 else "NO - Low Stroke Risk"

    print(f"--- Patient {i} ---")
    print(f"  Gender         : {patient['gender']}")
    print(f"  Age Group      : {patient['age_group']}")
    print(f"  Hypertension   : {patient['hypertension']}")
    print(f"  Heart Disease  : {patient['heart_disease']}")
    print(f"  Ever Married   : {patient['ever_married']}")
    print(f"  Work Type      : {patient['work_type']}")
    print(f"  Residence      : {patient['Residence_type']}")
    print(f"  Smoking Status : {patient['smoking_status']}")
    print(f"  Glucose Level  : {patient['glucose_level']}")
    print(f"  BMI Category   : {patient['bmi_category']}")
    print(f"  Stroke Prediction : {result}")
    print(f"  Probability YES   : {probability[1]*100:.1f}%")
    print(f"  Probability NO    : {probability[0]*100:.1f}%")
    print()

--- Patient 1 ---
  Gender         : Male
  Age Group      : Senior
  Hypertension   : Yes
  Heart Disease  : Yes
  Ever Married   : Yes
  Work Type      : Private
  Residence      : Urban
  Smoking Status : smokes
  Glucose Level  : High
  BMI Category   : Obese
  Stroke Prediction : YES - Stroke Risk
  Probability YES   : 92.7%
  Probability NO    : 7.3%

--- Patient 2 ---
  Gender         : Female
  Age Group      : Young
  Hypertension   : No
  Heart Disease  : No
  Ever Married   : No
  Work Type      : Private
  Residence      : Rural
  Smoking Status : never smoked
  Glucose Level  : Normal
  BMI Category   : Normal
  Stroke Prediction : NO - Low Stroke Risk
  Probability YES   : 0.0%
  Probability NO    : 100.0%

--- Patient 3 ---
  Gender         : Male
  Age Group      : Middle-aged
  Hypertension   : No
  Heart Disease  : No
  Ever Married   : Yes
  Work Type      : Self-employed
  Residence      : Urban
  Smoking Status : formerly smoked
  Glucose Level  : Normal
  BMI Cate

This model takes patient health information such as age, BMI, glucose level, smoking status, and medical history, and trains a Naive Bayes Categorical model to predict whether a patient is at risk of having a stroke. This could be useful, as strokes are a serious and often preventable medical event. With this model, a patient's health profile can be checked and accurately classified as high risk or low risk. Healthcare providers can use these predictions to identify at-risk patients earlier and take preventive action before a stroke actually happens.